# Working Prototype: REDFIN WebScraper

In [ ]:
!/opt/anaconda3/bin/python -m pip install playwright beautifulsoup4 pandas
!/opt/anaconda3/bin/python -m playwright install chromium
%pip install -U playwright-stealth

import asyncio
import json
import os
import random
import re

import nest_asyncio
import pandas as pd
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
from playwright_stealth import Stealth

nest_asyncio.apply()

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
BASE_URL = "https://www.redfin.com/city/30756/GA/Atlanta/rentals"
MAX_PAGES = None               # None = scrape every page; set an int (e.g. 3) to cap it for testing
START_PAGE = 1                 # First page to fetch from the network (earlier pages are loaded from disk if saved)
HEADLESS = True                # Set False if you get blocked

# Pacing / anti-block
DELAY_RANGE = (10.0, 20.0)     # Seconds between page requests
BLOCK_RETRIES = 3              # Retries of the SAME page after a block
BLOCK_COOLDOWN = (240.0, 360.0)  # Seconds to wait before each retry (about 4-6 minutes)
BATCH_SIZE = 5                 # Pages fetched per batch before a long rest
BATCH_REST = (180.0, 300.0)    # Seconds to rest between batches (about 3-5 minutes)
BLOCKED_RESOURCE_TYPES = {"image", "media", "font"}   # Not needed: we only read the ld+json script

# Save / resume
PROGRESS_DIR = "redfin_progress"   # Each page is saved here as page_001.csv, page_002.csv, ...
RESUME = True                      # Skip pages that already have a saved file

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)
LAUNCH_ARGS = [
    "--disable-blink-features=AutomationControlled",
    "--no-sandbox",
    "--disable-dev-shm-usage",
]
BLOCK_TITLE_KEYWORDS = ["robot", "access denied", "verify", "captcha", "error", "just a moment"]
BLOCK_BODY_MARKERS = ["the request could not be satisfied", "verify you are a human", "captcha"]

COLUMNS = ["Address", "City", "State", "Postal Code", "latitude", "longitude", "url", "Rental Price"]


# ---------------------------------------------------------------------------
# JSON-LD parsing
# ---------------------------------------------------------------------------
def first_number(value):
    """'1249' -> 1249.0, '1,249' -> 1249.0, '1249-1800' -> 1249.0 (low end), else None."""
    if value is None:
        return None
    match = re.search(r"\d[\d,]*(?:\.\d+)?", str(value))
    return float(match.group(0).replace(",", "")) if match else None

def iter_ld_objects(soup):
    """Yield every dict found in every <script type="application/ld+json"> block."""
    for script in soup.find_all("script", type="application/ld+json"):
        raw = script.string or script.get_text()
        if not raw or not raw.strip():
            continue
        try:
            data = json.loads(raw)
        except json.JSONDecodeError:
            continue
        for obj in (data if isinstance(data, list) else [data]):
            if isinstance(obj, dict):
                yield obj

def extract_offer_price(offers):
    """offers may be a dict or a list of dicts; price may sit under price/lowPrice."""
    for offer in (offers if isinstance(offers, list) else [offers]):
        if isinstance(offer, dict):
            price = first_number(offer.get("price", offer.get("lowPrice")))
            if price is not None:
                return price
    return None

def parse_listings(html):
    """
    Each listing appears as two JSON objects sharing the same 'url':
      - an Accommodation (address, geo, size)
      - a Product (offers -> price)
    Merge them on url.
    """
    soup = BeautifulSoup(html, "html.parser")
    merged = {}

    for obj in iter_ld_objects(soup):
        url = obj.get("url")
        if not url:
            continue
        rec = merged.setdefault(url, {"url": url})

        addr = obj.get("address")
        if isinstance(addr, dict):
            rec["Address"] = addr.get("streetAddress")
            rec["City"] = addr.get("addressLocality")
            rec["State"] = addr.get("addressRegion")
            rec["Postal Code"] = addr.get("postalCode")

        geo = obj.get("geo")
        if isinstance(geo, dict):
            # float() directly so negative longitudes keep their sign
            try:
                rec["latitude"] = float(geo.get("latitude"))
                rec["longitude"] = float(geo.get("longitude"))
            except (TypeError, ValueError):
                pass

        if "offers" in obj:
            price = extract_offer_price(obj["offers"])
            if price is not None:
                rec["Rental Price"] = price

    return list(merged.values())

def get_total_pages(html):
    """Read N from <span class="pageText" ...>Viewing page 1 of N</span>."""
    soup = BeautifulSoup(html, "html.parser")
    span = soup.select_one('span.pageText[data-rf-test-name="download-and-save-page-number-text"]')
    if span is None:
        span = soup.select_one("span.pageText")
    if span is None:
        return None
    match = re.search(r"of\s+(\d+)", span.get_text(" ", strip=True))
    return int(match.group(1)) if match else None


# ---------------------------------------------------------------------------
# Save / resume helpers
# ---------------------------------------------------------------------------
def page_path(page_num):
    return os.path.join(PROGRESS_DIR, f"page_{page_num:03d}.csv")

def meta_path():
    return os.path.join(PROGRESS_DIR, "meta.json")

def save_page(page_num, df_page):
    os.makedirs(PROGRESS_DIR, exist_ok=True)
    df_page.to_csv(page_path(page_num), index=False)

def load_page(page_num):
    """Return the saved DataFrame for a page, or None if it isn't on disk."""
    path = page_path(page_num)
    if not os.path.exists(path):
        return None
    return pd.read_csv(path, dtype={"Postal Code": str})

def save_total_pages(total_pages):
    os.makedirs(PROGRESS_DIR, exist_ok=True)
    with open(meta_path(), "w") as f:
        json.dump({"total_pages": total_pages}, f)

def load_total_pages():
    try:
        with open(meta_path()) as f:
            return int(json.load(f)["total_pages"])
    except Exception:
        return None


# ---------------------------------------------------------------------------
# Fetching
# ---------------------------------------------------------------------------
async def is_blocked(page, html):
    title = (await page.title()).lower()
    if any(k in title for k in BLOCK_TITLE_KEYWORDS):
        return True
    lowered = html.lower()
    return any(m in lowered for m in BLOCK_BODY_MARKERS)

async def block_heavy_resources(route):
    """Skip images / media / fonts to cut the number of requests per page."""
    if route.request.resource_type in BLOCKED_RESOURCE_TYPES:
        await route.abort()
    else:
        await route.continue_()

async def scrape_single_page(url, need_total=False):
    """
    The original fetch logic for ONE page: fresh Playwright session -> browser -> page -> parse -> close.
    Returns (df_page, total_pages, status, title) where status is "OK", "BLOCKED" or "EMPTY".
    """
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=HEADLESS, args=LAUNCH_ARGS)
        context = await browser.new_context(
            user_agent=USER_AGENT,
            viewport={"width": 1440, "height": 900},
            locale="en-US",
            timezone_id="America/New_York",
            extra_http_headers={"Accept-Language": "en-US,en;q=0.9"},
        )
        await Stealth().apply_stealth_async(context)
        await context.route("**/*", block_heavy_resources)
        page = await context.new_page()

        await page.goto(url, wait_until="domcontentloaded", timeout=30000)
        try:
            await page.wait_for_selector(
                'script[type="application/ld+json"]', state="attached", timeout=15000
            )
        except Exception:
            print("   Warning: no ld+json script appeared within timeout.")

        if need_total:
            try:
                await page.wait_for_selector("span.pageText", state="attached", timeout=10000)
            except Exception:
                pass

        # Light scroll in case more cards/scripts are injected lazily
        for _ in range(3):
            await page.mouse.wheel(0, 1500)
            await asyncio.sleep(0.6)

        html = await page.content()
        title = await page.title()
        blocked = await is_blocked(page, html)

        await browser.close()

    if blocked:
        return pd.DataFrame(columns=COLUMNS), None, "BLOCKED", title

    rows = parse_listings(html)
    df_page = pd.DataFrame(rows)
    for col in COLUMNS:
        if col not in df_page.columns:
            df_page[col] = pd.NA
    df_page = df_page[COLUMNS]

    status = "OK" if len(df_page) else "EMPTY"
    return df_page, get_total_pages(html), status, title

async def scrape_redfin_rentals():
    page_frames = []                                   # one DataFrame per page, in page order
    total_pages = load_total_pages() if RESUME else None   # None until we read it from a fetched page
    fetched_this_run = 0

    for page_num in range(1, 10_000):
        if total_pages is not None and page_num > total_pages:
            break
        if MAX_PAGES is not None and page_num > MAX_PAGES:
            print(f"   Reached MAX_PAGES cap ({MAX_PAGES}). Stopping.")
            break

        # Reuse pages already saved on disk (resume), or skip pages before START_PAGE
        saved = load_page(page_num) if (RESUME or page_num < START_PAGE) else None
        if saved is not None and len(saved):
            print(f"--- Page {page_num}: loaded {len(saved)} listings from disk ---")
            page_frames.append(saved)
            continue
        if page_num < START_PAGE:
            continue

        # Page 1 uses BASE_URL; page n uses BASE_URL/page-n
        url = BASE_URL if page_num == 1 else f"{BASE_URL}/page-{page_num}"
        print(f"--- Fetching page {page_num} of {total_pages or '?'}: {url} ---")

        for attempt in range(BLOCK_RETRIES + 1):
            df_page, tp, status, title = await scrape_single_page(url, need_total=(total_pages is None))
            if status != "BLOCKED":
                break
            if attempt < BLOCK_RETRIES:
                wait = random.uniform(*BLOCK_COOLDOWN)
                print(f"   BLOCKED (title: '{title}'). Cooling down {wait/60:.1f} min, retry {attempt + 1}/{BLOCK_RETRIES}...")
                await asyncio.sleep(wait)

        if status == "BLOCKED":
            print(f"   BLOCKED after {BLOCK_RETRIES} retries (title: '{title}'). Stopping.")
            print(f"   Progress is saved in '{PROGRESS_DIR}'. Rerun later to resume from page {page_num}.")
            break

        if total_pages is None:
            if tp is None:
                print("   Warning: could not find the total page count. Scraping this page only.")
                total_pages = page_num
            else:
                total_pages = tp
                save_total_pages(total_pages)
                print(f"   Total pages detected: {total_pages}")

        print(f"   Parsed {len(df_page)} listings.")
        if status == "EMPTY":
            print(f"   FAILED to find listings (title: '{title}').")
            break

        save_page(page_num, df_page)      # write this page to disk right away
        page_frames.append(df_page)       # and add it to the collection
        fetched_this_run += 1

        last_page = total_pages if MAX_PAGES is None else min(total_pages, MAX_PAGES)
        if page_num < last_page:
            if fetched_this_run % BATCH_SIZE == 0:
                rest = random.uniform(*BATCH_REST)
                print(f"   Batch of {BATCH_SIZE} done. Resting {rest/60:.1f} min...")
                await asyncio.sleep(rest)
            else:
                await asyncio.sleep(random.uniform(*DELAY_RANGE))

    if not page_frames:
        return pd.DataFrame(columns=COLUMNS)

    # Final DataFrame = every page stacked together
    df = pd.concat(page_frames, ignore_index=True)
    df = df.drop_duplicates(subset="url").reset_index(drop=True)
    return df

# Run in Jupyter (top-level await)
df_rentals = await scrape_redfin_rentals()



# ---------------------------------------------------------------------------
# Final Ouput
# ---------------------------------------------------------------------------

print("\n--- Final Dataset Summary ---")
print(df_rentals.head(10))
print(f"\nRows: {len(df_rentals)} | missing Rental Price: {df_rentals['Rental Price'].isna().sum()}")

import os
from datetime import datetime

def save_rentals_version(df): 
    folder = "redfin_rentals_versions"   # change this to any folder name or full path
    os.makedirs(folder, exist_ok=True)

    timestamp = datetime.now().strftime("%Y-%m-%d_%H%M")
    filename = os.path.join(folder, f"redfin_atlanta_rentals_{timestamp}.csv")

    df_rentals.to_csv(filename, index=False)
    print(f"Saved {len(df_rentals)} rows to {filename}")

# Saving version of scraped data to designated location
save_rentals_version(df_rentals)
df_rentals

--- Fetching page 1 of ?: https://www.redfin.com/city/30756/GA/Atlanta/rentals ---
   Total pages detected: 71
   Parsed 42 listings.
--- Fetching page 2 of 71: https://www.redfin.com/city/30756/GA/Atlanta/rentals/page-2 ---
   Parsed 42 listings.
--- Fetching page 3 of 71: https://www.redfin.com/city/30756/GA/Atlanta/rentals/page-3 ---
   Parsed 42 listings.
--- Fetching page 4 of 71: https://www.redfin.com/city/30756/GA/Atlanta/rentals/page-4 ---
   Parsed 42 listings.
--- Fetching page 5 of 71: https://www.redfin.com/city/30756/GA/Atlanta/rentals/page-5 ---
   Parsed 42 listings.
   Batch of 5 done. Resting 4.3 min...
--- Fetching page 6 of 71: https://www.redfin.com/city/30756/GA/Atlanta/rentals/page-6 ---
   Parsed 42 listings.
--- Fetching page 7 of 71: https://www.redfin.com/city/30756/GA/Atlanta/rentals/page-7 ---
   BLOCKED (title: 'ERROR: The request could not be satisfied'). Cooling down 4.5 min, retry 1/3...
   Parsed 42 listings.
--- Fetching page 8 of 71: https://www.redf

,Address,City,State,Postal Code,latitude,longitude,url,Rental Price
0,NaN,NaN,NaN,NaN,NaN,NaN,https://www.redfin.com,NaN
1,1760 Northside Dr NW,Atlanta,GA,30318,33.803653,-84.408935,https://www.redfin.com/GA/Atlanta/Castlegate-C...,1249.0
2,6640 Embassy Ct NW,Sandy Springs,GA,30328,33.914955,-84.389336,https://www.redfin.com/GA/Sandy-Springs/Envoy/...,1763.0
3,1400 Lake Hearn Dr NE,Atlanta,GA,30319,33.914896,-84.344573,https://www.redfin.com/GA/Atlanta/AMLI-Brookha...,1891.0
4,1950 Howell Mill Rd NW,Atlanta,GA,30318,33.807954,-84.412947,https://www.redfin.com/GA/Atlanta/Bell-Collier...,1215.0
...,...,...,...,...,...,...,...,...
2682,1159 Thornwell Dr NE Unit 1159,Atlanta,GA,30319,33.854653,-84.341859,https://www.redfin.com/GA/Atlanta/1159-Thornwe...,7500.0
2683,2124 Gables Dr NE Unit 2117,Atlanta,GA,30319,33.846580,-84.344017,https://www.redfin.com/GA/Atlanta/2124-Gables-...,1784.0
2684,41 25th St NW Apt 8,Atlanta,GA,30309,33.801534,-84.394473,https://www.redfin.com/GA/Atlanta/41-25th-St-N...,1495.0
2685,892 Ambient Way Unit 892,Atlanta,GA,30331,33.746552,-84.502604,https://www.redfin.com/GA/Atlanta/892-Ambient-...,2300.0


# Save Data

In [13]:
save_rentals_version(df_rentals)

Saved 2687 rows to redfin_rentals_versions/redfin_atlanta_rentals_2026-10-14_1743.csv


# Reload Data

In [16]:
import glob
import pandas as pd

def load_latest_rentals():
    files = sorted(glob.glob("redfin_rentals_versions/redfin_atlanta_rentals_*.csv"))
    if files:
        latest = files[-1]
        df = pd.read_csv(latest, dtype={"Postal Code": str})
        print(f"Loaded {latest} ({len(df)} rows)")
        return df
    else:
        print("No saved versions found yet. Run the save cell first.")
        return None


df_rentals = load_latest_rentals()
df_rentals

Loaded redfin_rentals_versions/redfin_atlanta_rentals_2026-10-14_1742.csv (2687 rows)


,Address,City,State,Postal Code,latitude,longitude,url,Rental Price
0,NaN,NaN,NaN,NaN,NaN,NaN,https://www.redfin.com,NaN
1,1760 Northside Dr NW,Atlanta,GA,30318,33.803653,-84.408935,https://www.redfin.com/GA/Atlanta/Castlegate-C...,1249.0
2,6640 Embassy Ct NW,Sandy Springs,GA,30328,33.914955,-84.389336,https://www.redfin.com/GA/Sandy-Springs/Envoy/...,1763.0
3,1400 Lake Hearn Dr NE,Atlanta,GA,30319,33.914896,-84.344573,https://www.redfin.com/GA/Atlanta/AMLI-Brookha...,1891.0
4,1950 Howell Mill Rd NW,Atlanta,GA,30318,33.807954,-84.412947,https://www.redfin.com/GA/Atlanta/Bell-Collier...,1215.0
...,...,...,...,...,...,...,...,...
2682,1159 Thornwell Dr NE Unit 1159,Atlanta,GA,30319,33.854653,-84.341859,https://www.redfin.com/GA/Atlanta/1159-Thornwe...,7500.0
2683,2124 Gables Dr NE Unit 2117,Atlanta,GA,30319,33.846580,-84.344017,https://www.redfin.com/GA/Atlanta/2124-Gables-...,1784.0
2684,41 25th St NW Apt 8,Atlanta,GA,30309,33.801534,-84.394473,https://www.redfin.com/GA/Atlanta/41-25th-St-N...,1495.0
2685,892 Ambient Way Unit 892,Atlanta,GA,30331,33.746552,-84.502604,https://www.redfin.com/GA/Atlanta/892-Ambient-...,2300.0
